# Lab 6 · Dữ liệu ngoài: file lớn, API lịch sử & DuckDB

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 6**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 6 gọi API dự báo thời tiết. Lab này đi xa hơn: đọc **bảng đầy đủ 90 cột**
cho khéo, lấy **thời tiết quá khứ** rồi ghép với nhịp review theo ngày, và để DuckDB cân
690 nghìn dòng review — có đối chiếu chéo với con số bạn đã tính ở lab 2.

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — micro-exercise 🔒
  cuối giờ đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Đọc file lớn có chọn lọc bằng `usecols` + `parse_dates` và thấy lợi ích đo được.
2. Ghi Parquet và so kích thước/kiểu dữ liệu với CSV.
3. Gọi một API lịch sử (archive), lưu response thô, ghép với dữ liệu review theo ngày.
4. Viết truy vấn DuckDB có `JOIN` — và kiểm chứng chéo bằng con số đã biết.

In [ ]:
%pip install -q duckdb

## Phần 0 · Khởi động (~8 phút)

In [ ]:
import pandas as pd

# W1 — đọc reviews.csv (2 cột) với parse_dates ngay từ lúc đọc
URL_RV = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/visualisations/reviews.csv")
# TODO: đọc URL_RV với parse_dates=["date"] vào biến rv
rv = ...

# --- Ô kiểm tra ---
assert len(rv) == 690112
assert str(rv["date"].dtype).startswith("datetime64")
print("W1 ổn: 690.112 review, cột date đã là datetime ngay từ lúc đọc.")

## Phần 1 · Bảng đầy đủ 90 cột — đọc có chọn lọc (~20 phút)

Đến giờ ta dùng bản rút gọn 19 cột. Bảng **đầy đủ** (`data/listings.csv.gz`) có 90 cột —
và cột giá ở đây là chuỗi `"$45,647.00"` đúng như bài giảng cảnh báo.

In [ ]:
import time

URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")

# Cách 1: đọc cả bảng (đo thời gian) — ô này cho sẵn
t0 = time.perf_counter()
full = pd.read_csv(URL_FULL)
t_full = time.perf_counter() - t0
print(f"Cả bảng : {full.shape} — {t_full:.1f}s")

# TODO Cách 2: chỉ đọc 5 cột ["id", "room_type", "price", "first_review",
#      "number_of_reviews"], parse_dates cho first_review
t0 = time.perf_counter()
nho = ...
t_nho = time.perf_counter() - t0
print(f"5 cột   : {nho.shape} — {t_nho:.1f}s")

# --- Ô kiểm tra ---
assert full.shape == (18534, 90) and nho.shape == (18534, 5)
assert str(nho["first_review"].dtype).startswith("datetime64")
print("Cùng file, đọc đúng thứ cần — nhẹ và nhanh hơn hẳn.")

In [ ]:
# TODO: làm sạch cột giá chuỗi "$45,647.00" -> float (2 lần str.replace + astype)
nho["price_num"] = ...

# TODO: ghi nho ra 2 định dạng rồi so kích thước file
nho.to_csv("t6.csv", index=False)
nho.to_parquet("t6.parquet")
import os
size_csv = ...
size_pq = ...

# --- Ô kiểm tra ---
assert nho["price_num"].median() == 59000.0
assert size_pq < size_csv * 0.6, "Parquet phải nhỏ hơn hẳn CSV"
print(f"CSV {size_csv/1e6:.1f} MB — Parquet {size_pq/1e6:.1f} MB; và Parquet giữ nguyên dtype datetime.")

## Phần 2 · API lịch sử + ghép với nhịp review (~25 phút)

Câu hỏi: *tháng 1/2025 (giữa hè Santiago), ngày nóng hơn có nhiều review hơn không?*
Cần hai nguồn: thời tiết **quá khứ** (API archive của Open-Meteo — dữ liệu lịch sử nên
chạy lại luôn ra đúng kết quả) và số review theo ngày (từ `rv`).

In [ ]:
import requests, json
from pathlib import Path

r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={"latitude": -33.45, "longitude": -70.66,
            "start_date": "2025-01-01", "end_date": "2025-01-31",
            "daily": "temperature_2m_max", "timezone": "auto"},
    timeout=30,
)
r.raise_for_status()

# Lưu response THÔ kèm mốc thời gian — thói quen "giữ bằng chứng"
Path("raw").mkdir(exist_ok=True)
Path("raw/weather_scl_2025-01.json").write_text(json.dumps(r.json()), encoding="utf-8")

thoi_tiet = pd.DataFrame(r.json()["daily"])
thoi_tiet.head(3)

In [ ]:
# --- Ô kiểm tra dữ liệu thời tiết ---
assert len(thoi_tiet) == 31
tb_max = thoi_tiet["temperature_2m_max"].mean()
assert 27 <= tb_max <= 34, f"Nhiệt độ TB tháng 1 Santiago ~30-31°C, đang ra {tb_max:.1f}"
print(f"31 ngày, nhiệt độ cao nhất trung bình {tb_max:.1f}°C — đúng mùa hè nam bán cầu.")

In [ ]:
# TODO: đếm số review MỖI NGÀY của tháng 1/2025 từ rv
#   1) lọc rv trong khoảng 2025-01-01 .. 2025-01-31
#   2) groupby theo chuỗi ngày (rv["date"].dt.strftime("%Y-%m-%d")), size()
#   3) đưa về DataFrame 2 cột: time, so_review  (reset_index + đặt tên)
jan = ...
theo_ngay = ...

# --- Ô kiểm tra ---
assert len(jan) == 14636 and len(theo_ngay) == 31
assert list(theo_ngay.columns) == ["time", "so_review"]
theo_ngay.head(3)

In [ ]:
# TODO: merge thoi_tiet với theo_ngay theo cột "time" (how="left")
gop = ...

# --- Ô kiểm tra ---
assert gop.shape == (31, 3) and gop["so_review"].isna().sum() == 0

# Tương quan ngày-nóng vs ngày-nhiều-review:
he_so = gop["temperature_2m_max"].corr(gop["so_review"])
print(f"Hệ số tương quan: {he_so:.3f}")

Hệ số ~0.08 — gần như **không có** liên hệ ngày-nóng ↔ ngày-nhiều-review.
Đây là kết quả thật và đáng giá: đặt giả thuyết → ghép dữ liệu → đo → *chấp nhận kết quả
"không thấy gì"*. Nhịp review theo ngày phụ thuộc lịch trả phòng và thói quen viết review
nhiều hơn thời tiết của đúng hôm đó. Một phân tích trung thực được phép kết luận "không".

## Phần 3 · DuckDB trên 690 nghìn dòng (~15 phút)

DuckDB query thẳng file — kể cả file to. Trước hết tải hai file về máy Colab để SQL trỏ vào.

In [ ]:
import urllib.request
import duckdb

urllib.request.urlretrieve(URL_RV, "reviews.csv")
urllib.request.urlretrieve(
    "https://data.insideairbnb.com/chile/rm/santiago/"
    "2026-06-29/visualisations/listings.csv", "listings.csv")
print("Đã tải 2 file.")

In [ ]:
# TODO: viết truy vấn DuckDB đếm số review THEO NĂM, trực tiếp trên 'reviews.csv'
#   SELECT year(date) AS nam, count(*) AS n FROM ... GROUP BY ... ORDER BY nam
dem_nam = duckdb.query(...).df()

# --- Ô kiểm tra: đối chiếu chéo với con số bạn đã tính bằng Python thuần ở lab 2 ---
nam_2025 = int(dem_nam.loc[dem_nam["nam"] == 2025, "n"].iloc[0])
assert nam_2025 == 211432, "Phải khớp đúng con số đếm bằng dict ở lab 2!"
print("SQL và Python thuần — hai công cụ độc lập, một đáp số: 211.432 review năm 2025.")

In [ ]:
# TODO: JOIN hai file — đếm review năm 2025 theo loại phòng
#   FROM 'reviews.csv' r JOIN 'listings.csv' l ON r.listing_id = l.id
#   WHERE year(r.date) = 2025, GROUP BY room_type, ORDER BY n giảm dần
theo_loai = duckdb.query(...).df()

# --- Ô kiểm tra ---
assert theo_loai.iloc[0]["room_type"] == "Entire home/apt"
assert int(theo_loai.iloc[0]["n"]) == 192326
theo_loai

Nguyên căn hút 192 nghìn / 211 nghìn review 2025 (91%) — cao hơn cả tỷ trọng listing
của nó (81%). JOIN + WHERE + GROUP BY trong một câu — với file 690 nghìn dòng, gần như tức thì.

## Phần 4 · Bài tự làm 🔓 (làm xong sớm / về nhà)

Được dùng AI theo quy trình 5 bước; ghi lại prompt + cách kiểm chứng.

### Tự làm 1 · Mùa đông thì sao?

Lặp Phần 2 cho **tháng 7/2025** (giữa đông): gọi archive API, đếm review theo ngày, merge,
tính tương quan. So với tháng 1 và viết 2 câu nhận xét. (Nhớ lưu response thô ra `raw/`.)

### Tự làm 2 · SQL truy tìm

Viết truy vấn DuckDB tìm **5 listing nhiều review nhất năm 2025** kèm tên (JOIN lấy `name`).
Kiểm chứng con số đầu bảng bằng pandas (`rv` + lọc năm + `value_counts` trên listing_id`).

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Bạn đã làm | Sẽ gặp lại ở |
|---|---|
| usecols/parse_dates trên bảng 90 cột | script thu thập của bài tập lớn |
| Parquet nhỏ hơn ~3 lần, giữ dtype | dữ liệu trung gian `processed/` |
| API lịch sử + lưu response thô | mọi nguồn API; tinh thần cache buổi 11 |
| Ghép 2 nguồn theo ngày, chấp nhận "không thấy tương quan" | trung thực phân tích (buổi 14) |
| DuckDB JOIN 690k dòng + đối chiếu chéo lab 2 | kiểm chứng chéo hai công cụ |

Buổi lý thuyết tới: xử lý **chuỗi và regex** — cột văn bản bắt đầu lên tiếng.